In [ ]:
SELECT table_name, column_name
FROM spark_catalog.silver.information_schema.columns
WHERE lower(column_name) LIKE '%service%'
   OR lower(column_name) LIKE '%clienttype%'
   OR lower(column_name) LIKE '%tenancy%'
ORDER BY table_name, column_name;

In [ ]:
SELECT DISTINCT trim(clienttype) AS v
FROM <that_table>
WHERE clienttype IS NOT NULL AND trim(clienttype) <> ''
LIMIT 50;

In [ ]:
# Search columns across all tables in a database (Lakehouse schema)
db = "silver"
patterns = ["service", "clienttype", "tenancy"]

tables = [t.name for t in spark.catalog.listTables(db) if t.tableType.lower() != "view"]
hits = []

for tbl in tables:
    cols = [c.name.lower() for c in spark.table(f"{db}.{tbl}").schema.fields]
    if any(any(p in col for p in patterns) for col in cols):
        for c in cols:
            if any(p in c for p in patterns):
                hits.append((tbl, c))

display(spark.createDataFrame(hits, ["table_name", "column_name"]).orderBy("table_name", "column_name"))